In [1]:
import pickle 
import mlflow 
import mlflow.sklearn


import pandas as pd 

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


from sklearn.pipeline import make_pipeline

In [2]:
import os

os.environ["AWS_ACCESS_KEY_ID"] = "ALG"
os.environ["AWS_SECRET_ACCESS_KEY"] = "wJ"
os.environ["AWS_DEFAULT_REGION"] = "eu-north-1"


In [4]:
import boto3
s3 = boto3.client("s3")
print([b["Name"] for b in s3.list_buckets()["Buckets"]])


['s3-bucket-default-mlflow']


In [6]:
import mlflow 

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("green-taxi-duration1")

<Experiment: artifact_location='s3://s3-bucket-default-mlflow/2', creation_time=1759829720602, experiment_id='2', last_update_time=1759829720602, lifecycle_stage='active', name='green-taxi-duration1', tags={}>

In [8]:
def read_dataframe(filename:str):
    df=pd.read_parquet(filename)
    
    df['duration']=df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical=['PULocationID', 'DOLocationID']
    df[categorical]=df[categorical].astype(str)
    return df


In [9]:
def prepare_dict(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical=["PU_DO"]
    numerical=["trip_distance"]
    dicts=df[categorical + numerical].to_dict(orient='records')
    return dicts

In [10]:
df_train=read_dataframe("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
df_val=read_dataframe("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")

target="duration"
y_train=df_train[target].values
y_val=df_val[target].values

dict_train=prepare_dict(df_train)
dict_val=prepare_dict(df_val)

In [11]:
with mlflow.start_run():
    params=dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=13)
    mlflow.log_params(params)
    
    dv=DictVectorizer()
    model=RandomForestRegressor(**params, n_jobs=-1)
    
    X_train= dv.fit_transform(dict_train)
    model.fit(X_train, y_train)
    
    X_val= dv.transform(dict_val)
    y_pred=model.predict(X_val)
    
    rmse=mean_squared_error(y_pred, y_val, squared=False)
    print(params, rmse)
    mlflow.log_metric("rmse", rmse)

    #os.makedirs("preprocessor", exist_ok=True)
    with open("dict_vectorizer.bin", 'wb') as f_out:
        pickle.dump(dv, f_out)
    
    #mlflow.log_artifact("dict_vectorizer.bin", artifact_path="preprocessor")
    
    mlflow.sklearn.log_model(model, artifact_path="model", code_paths=['preprocessor'])
    
    
    

2025/10/08 07:09:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 13} 6.756305038881697


2025/10/08 07:09:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run kindly-moth-925 at: http://127.0.0.1:5000/#/experiments/2/runs/73d41ed716a54e14b1dcf2bec49a8e1d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [12]:
import mlflow.sklearn
run_id="6b670d2865fe4e42bd49e04afaa6c113"
logged_model=f"runs:/{run_id}/model"
loaded_model=mlflow.pyfunc.load_model(logged_model)

In [13]:
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: s3://s3-bucket-default-mlflow/2/models/m-3ae64fd9f91d4cdc86f0786fd2ec5ac1/artifacts
  flavor: mlflow.sklearn
  run_id: 6b670d2865fe4e42bd49e04afaa6c113

In [22]:
print(logged_model)

runs:/6b670d2865fe4e42bd49e04afaa6c113/model


In [ ]:
#------------------------------------------------------

In [18]:
dv_run_id="80d1a2a434b9493c84eff78c691f0700"

In [19]:
path=mlflow.artifacts.download_artifacts(run_id=dv_run_id, artifact_path="dict_vectorizer.bin")

In [20]:
with open(path, 'rb') as f_out:
    dv=pickle.load(f_out)

In [21]:
dv

DictVectorizer()

In [ ]:
#--------------------------------------

In [24]:
print(logged_model)
print(mlflow.get_tracking_uri())


runs:/6b670d2865fe4e42bd49e04afaa6c113/model
http://127.0.0.1:5000


In [25]:
#-------------------------------------------------

In [26]:
# create a pipeline for the model download and artifacts downloads 

In [27]:
from sklearn.pipeline import make_pipeline 

In [33]:
with mlflow.start_run():
    params=dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=13)
    mlflow.log_params(params)
    
    pipeline=make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred=pipeline.predict(dict_val)
    
    rmse=mean_squared_error(y_pred, y_val, squared=False)
    print(params, rmse)
    mlflow.log_metric("rmse", rmse)
    
    mlflow.sklearn.log_model(pipeline, artifact_path="model")
    

2025/10/08 07:50:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 13} 6.756305038881697


2025/10/08 07:50:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run ambitious-whale-527 at: http://127.0.0.1:5000/#/experiments/2/runs/18a0d886c5c5494abae629e03532c2fb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
